## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-26-17-54-43 +0000,bbc,Burnham says UK net migration now under 'suffi...,https://www.bbc.co.uk/news/articles/ck792l8ndq...
1,2026-11-26-17-53-04 +0000,nyt,European Central Bank Chooses Pablo Hernández ...,https://www.nytimes.com/2026/11/26/business/ec...
2,2026-11-26-17-41-14 +0000,nyt,"Jensen Huang Is Back, and So Is His Plaque at ...",https://www.nytimes.com/2026/11/26/business/de...
3,2026-11-26-17-40-41 +0000,wapo,"As he races to unseat Netanyahu, Eisenkot reac...",https://www.washingtonpost.com/world/2026/11/2...
4,2026-11-26-17-38-56 +0000,bbc,"More than 1,000 flights cancelled or delayed i...",https://www.bbc.co.uk/news/articles/ckm2mz3prq...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2764/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
100,trump,27
38,thanksgiving,26
263,new,20
106,china,13
23,he,13


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
83,2026-11-26-13-58-19 +0000,nypost,Trump kicks off Thanksgiving with late night d...,https://nypost.com/2026/11/26/us-news/presiden...,73
45,2026-11-26-15-39-23 +0000,nypost,Trump tells Putin to ‘pull back all military a...,https://nypost.com/2026/11/26/us-news/trump-te...,72
207,2026-11-26-05-04-49 +0000,nypost,Russia must avoid ‘any actions’ expanding war ...,https://nypost.com/2026/11/26/us-news/russia-m...,69
215,2026-11-26-04-14-04 +0000,nypost,Abdul El-Sayed rips Trump — in Thanksgiving me...,https://nypost.com/2026/11/25/us-news/abdul-el...,68
341,2026-11-25-19-25-40 +0000,nypost,NATO boss says Ukraine ‘doing exceptionally we...,https://nypost.com/2026/11/25/us-news/nato-bos...,67


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
83,73,2026-11-26-13-58-19 +0000,nypost,Trump kicks off Thanksgiving with late night d...,https://nypost.com/2026/11/26/us-news/presiden...
164,61,2026-11-26-10-00-00 +0000,wsj,Democratic Sen. Chris Van Hollen wants to be S...,https://www.wsj.com/politics/policy/why-this-d...
266,57,2026-11-25-22-38-54 +0000,nypost,Elon Musk’s plan for SpaceX to launch 1 millio...,https://nypost.com/2026/11/25/business/fcc-app...
148,54,2026-11-26-10-13-24 +0000,nypost,Hong Kong marks anniversary high-rise blaze th...,https://nypost.com/2026/11/26/world-news/hong-...
276,52,2026-11-25-22-21-10 +0000,cbc,"Andy Taylor, guitarist of new wave icons Duran...",https://www.cbc.ca/news/entertainment/duran-du...
104,41,2026-11-26-12-42-00 +0000,wsj,A Game-Changing Move Just Gave America the Edg...,https://www.wsj.com/tech/a-game-changing-move-...
368,40,2026-11-25-18-10-53 +0000,nypost,PETA has beef with Nvidia boss Jensen Huang’s ...,https://nypost.com/2026/11/25/business/peta-ha...
42,39,2026-11-26-15-45-19 +0000,nypost,Ex-NFL punter flips deep red California seat b...,https://nypost.com/2026/11/26/us-news/former-v...
207,38,2026-11-26-05-04-49 +0000,nypost,Russia must avoid ‘any actions’ expanding war ...,https://nypost.com/2026/11/26/us-news/russia-m...
10,36,2026-11-26-17-12-53 +0000,wapo,Two Italians killed in Kuwait as a strained Ir...,https://www.washingtonpost.com/world/2026/11/2...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
